# Alpaca RL Training on Kaggle GPU

This notebook trains a reinforcement learning agent for algorithmic trading using GPU acceleration.

**Setup:**
1. Enable GPU in Kaggle notebook settings
2. Add your dataset (exported from home lab)
3. Set environment variables for MinIO access
4. Run all cells

In [ ]:
# Install dependencies
!pip install -q stable-baselines3[extra] gymnasium ta scikit-learn boto3

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from datetime import datetime

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Configuration

Set these in Kaggle Secrets or as environment variables:

In [ ]:
# Training configuration
CONFIG = {
    "symbol": "SPY",
    "total_timesteps": 500_000,
    "trading_days": 252,
    "trading_cost_bps": 10,
    "time_cost_bps": 1,
    "learning_rate": 1e-4,
    "batch_size": 256,
    "buffer_size": 100_000,
    "architecture": [256, 256],
    "gamma": 0.99,
    "exploration_fraction": 0.3,
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,
}

# MinIO/S3 configuration (set in Kaggle Secrets)
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "")
S3_BUCKET = os.getenv("S3_BUCKET", "alpaca-rl-artifacts")
S3_ACCESS_KEY = os.getenv("S3_ACCESS_KEY", "")
S3_SECRET_KEY = os.getenv("S3_SECRET_KEY", "")

# Run ID for tracking
RUN_ID = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
print(f"Run ID: {RUN_ID}")

## Load Dataset

Dataset should be attached to this notebook from your Kaggle dataset upload.

In [ ]:
# Load data from Kaggle input
data_path = "/kaggle/input"  # Kaggle mounts datasets here

# Find the CSV file
import glob
csv_files = glob.glob(f"{data_path}/**/*.csv", recursive=True)
print(f"Found CSV files: {csv_files}")

if not csv_files:
    raise ValueError("No CSV file found in /kaggle/input. Make sure dataset is attached.")

df = pd.read_csv(csv_files[0])
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')

print(f"Loaded {len(df)} bars")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()

## Trading Environment

Copy of the trading environment from your home lab.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from sklearn.preprocessing import scale
import ta

class DataSource:
    FEATURE_COLS = [
        "returns", "ret_2", "ret_5", "ret_10", "ret_21",
        "rsi", "macd", "atr", "stoch", "ultosc",
    ]

    def __init__(self, df: pd.DataFrame, trading_days: int = 252, normalize: bool = True):
        self.trading_days = trading_days
        self.normalize = normalize
        self.data = self._preprocess(df)
        self.min_values = self.data.min()
        self.max_values = self.data.max()
        self.step = 0
        self.offset = None

    def _preprocess(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy().sort_index()
        
        df["returns"] = df["close"].pct_change()
        df["ret_2"]   = df["close"].pct_change(2)
        df["ret_5"]   = df["close"].pct_change(5)
        df["ret_10"]  = df["close"].pct_change(10)
        df["ret_21"]  = df["close"].pct_change(21)
        
        df["rsi"]    = ta.momentum.RSIIndicator(df["close"], window=14).rsi()
        macd_obj     = ta.trend.MACD(df["close"])
        df["macd"]   = macd_obj.macd_signal()
        df["atr"]    = ta.volatility.AverageTrueRange(
            df["high"], df["low"], df["close"], window=14
        ).average_true_range()
        stoch_obj    = ta.momentum.StochasticOscillator(
            df["high"], df["low"], df["close"], window=14
        )
        df["stoch"]  = stoch_obj.stoch_signal() - stoch_obj.stoch()
        df["ultosc"] = ta.momentum.UltimateOscillator(
            df["high"], df["low"], df["close"]
        ).ultimate_oscillator()
        
        df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=self.FEATURE_COLS)
        
        r = df["returns"].copy()
        if self.normalize:
            df[self.FEATURE_COLS] = scale(df[self.FEATURE_COLS])
        df["returns"] = r
        return df[self.FEATURE_COLS]

    def reset(self):
        high = len(self.data) - self.trading_days
        self.offset = np.random.randint(low=0, high=max(high, 1))
        self.step = 0

    def take_step(self):
        obs = self.data.iloc[self.offset + self.step].values
        market_return = self.data.iloc[self.offset + self.step]["returns"]
        self.step += 1
        done = self.step > self.trading_days
        return obs, market_return, done


class TradingSimulator:
    def __init__(self, steps: int, trading_cost_bps: float, time_cost_bps: float):
        self.trading_cost_bps = trading_cost_bps
        self.time_cost_bps = time_cost_bps
        self.steps = steps
        self.reset()

    def reset(self):
        self.step = 0
        self.actions         = np.zeros(self.steps)
        self.navs            = np.ones(self.steps)
        self.market_navs     = np.ones(self.steps)
        self.strategy_returns = np.zeros(self.steps)
        self.positions       = np.zeros(self.steps)
        self.costs           = np.zeros(self.steps)
        self.trades          = np.zeros(self.steps)
        self.market_returns  = np.zeros(self.steps)

    def take_step(self, action: int, market_return: float):
        start_position   = self.positions[max(0, self.step - 1)]
        start_nav        = self.navs[max(0, self.step - 1)]
        start_market_nav = self.market_navs[max(0, self.step - 1)]

        self.market_returns[self.step] = market_return
        self.actions[self.step] = action

        end_position = action - 1
        n_trades     = end_position - start_position
        self.positions[self.step] = end_position
        self.trades[self.step]    = n_trades

        trade_cost = abs(n_trades) * self.trading_cost_bps
        time_cost  = 0 if n_trades else self.time_cost_bps
        self.costs[self.step] = trade_cost + time_cost

        reward = start_position * market_return - self.costs[self.step]
        self.strategy_returns[self.step] = reward

        if self.step != 0:
            self.navs[self.step]        = start_nav * (1 + self.strategy_returns[self.step])
            self.market_navs[self.step] = start_market_nav * (1 + self.market_returns[self.step])

        self.step += 1
        return reward, {"nav": self.navs[self.step - 1], "costs": self.costs[self.step - 1]}


class TradingEnvironment(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, df: pd.DataFrame, trading_days: int = 252,
                 trading_cost_bps: float = 1e-3, time_cost_bps: float = 1e-4):
        super().__init__()
        self.trading_days     = trading_days
        self.trading_cost_bps = trading_cost_bps
        self.time_cost_bps    = time_cost_bps

        self.data_source = DataSource(df, trading_days=trading_days)
        self.simulator   = TradingSimulator(
            steps=trading_days,
            trading_cost_bps=trading_cost_bps,
            time_cost_bps=time_cost_bps,
        )

        n_features = len(DataSource.FEATURE_COLS)
        self.action_space      = spaces.Discrete(3)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(n_features,), dtype=np.float32
        )
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.data_source.reset()
        self.simulator.reset()
        obs, _, _ = self.data_source.take_step()
        return obs.astype(np.float32), {}

    def step(self, action: int):
        assert self.action_space.contains(action)
        obs, market_return, done = self.data_source.take_step()
        reward, info = self.simulator.take_step(
            action=action, market_return=market_return
        )
        return obs.astype(np.float32), float(reward), done, False, info

    def render(self):
        pass

print("Trading environment defined")

## Train Agent

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

class ProgressCallback(BaseCallback):
    def __init__(self, log_interval: int = 50):
        super().__init__()
        self.log_interval = log_interval
        self.episode_rewards = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:
                ep_reward = info["episode"]["r"]
                self.episode_rewards.append(ep_reward)
                
                if len(self.episode_rewards) % self.log_interval == 0:
                    recent = self.episode_rewards[-self.log_interval:]
                    avg_reward = np.mean(recent)
                    print(f"Episode {len(self.episode_rewards)}: avg_reward={avg_reward:.4f}")
        return True

# Create environment
raw_env = TradingEnvironment(
    df=df,
    trading_days=CONFIG["trading_days"],
    trading_cost_bps=CONFIG["trading_cost_bps"] / 10_000,
    time_cost_bps=CONFIG["time_cost_bps"] / 10_000,
)
env = Monitor(raw_env)

print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

# Create DQN agent
model = DQN(
    policy="MlpPolicy",
    env=env,
    learning_rate=CONFIG["learning_rate"],
    buffer_size=CONFIG["buffer_size"],
    batch_size=CONFIG["batch_size"],
    gamma=CONFIG["gamma"],
    exploration_fraction=CONFIG["exploration_fraction"],
    exploration_initial_eps=CONFIG["epsilon_start"],
    exploration_final_eps=CONFIG["epsilon_end"],
    policy_kwargs={"net_arch": CONFIG["architecture"]},
    verbose=1,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print(f"Training on device: {model.device}")
print(f"Total timesteps: {CONFIG['total_timesteps']:,}")

In [ ]:
# Train the agent
callback = ProgressCallback(log_interval=50)

print("Starting training...")
model.learn(
    total_timesteps=CONFIG["total_timesteps"],
    callback=callback,
    progress_bar=True
)

print("Training complete!")
print(f"Total episodes: {len(callback.episode_rewards)}")
print(f"Mean reward (last 50): {np.mean(callback.episode_rewards[-50:]):.4f}")

## Save Model

In [ ]:
# Save model locally
model_path = f"policy_{RUN_ID}"
model.save(model_path)
print(f"Model saved to {model_path}.zip")

# Save training metrics
metrics = {
    "run_id": RUN_ID,
    "config": CONFIG,
    "total_episodes": len(callback.episode_rewards),
    "mean_reward": float(np.mean(callback.episode_rewards[-50:])),
    "max_reward": float(np.max(callback.episode_rewards)),
    "episode_rewards": callback.episode_rewards[-100:],
}

with open(f"metrics_{RUN_ID}.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Metrics saved")

## Upload to MinIO (Optional)

If S3 credentials are configured, upload directly to your home lab.

In [ ]:
if S3_ENDPOINT and S3_ACCESS_KEY:
    import boto3
    
    s3 = boto3.client(
        "s3",
        endpoint_url=S3_ENDPOINT,
        aws_access_key_id=S3_ACCESS_KEY,
        aws_secret_access_key=S3_SECRET_KEY,
    )
    
    # Upload model
    s3_key = f"models/kaggle/{RUN_ID}/policy_best.zip"
    with open(f"{model_path}.zip", "rb") as f:
        s3.put_object(Bucket=S3_BUCKET, Key=s3_key, Body=f.read())
    
    print(f"Model uploaded to s3://{S3_BUCKET}/{s3_key}")
    
    # Upload metrics
    metrics_key = f"models/kaggle/{RUN_ID}/metrics.json"
    with open(f"metrics_{RUN_ID}.json", "rb") as f:
        s3.put_object(Bucket=S3_BUCKET, Key=metrics_key, Body=f.read())
    
    print(f"Metrics uploaded to s3://{S3_BUCKET}/{metrics_key}")
else:
    print("S3 credentials not configured. Download model manually from Kaggle output.")
    print(f"Model file: {model_path}.zip")
    print(f"Metrics file: metrics_{RUN_ID}.json")

## Done!

The trained model is now available:
- In Kaggle notebook output (download manually)
- In your MinIO bucket (if credentials were configured)

Next steps:
1. Download the model from Kaggle output
2. Upload to your home lab MinIO
3. Deploy for inference using the `rl-infer` service